In [2]:
# import libraries

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3



# 1.IMPORTING DATA 

In [16]:
# first of all take the name of excel file
excel_file_s="customer_churn_practice_expanded.xlsx"

# then create  connection python to sqlite3 by openinig database
connection=sqlite3.connect("customer_churn.db")

# now read excel file 
excel_data = pd.ExcelFile(excel_file_s)

#now read all the sheet  one by one and store it in data frame
for i in excel_data.sheet_names:

    df=pd.read_excel(excel_data,sheet_name=i) # read sheet in the data frame

    # now these sheet sote in the sqllite in the form of table
    df.to_sql(name=i,     # table name = sheet name
              con=connection,
              if_exists='replace',
              index=False
              )

connection.close()    

print("Now all the excel file conerted into sqllite  table Successfully")

Now all the excel file conerted into sqllite  table Successfully


In [17]:
# Data Import: db file to pandas, storing each table to a separate df

# Connect to SQLite database
conn = sqlite3.connect('customer_churn.db')

# sql query to Get all table names
sql_query = """
        SELECT name
        FROM sqlite_master
        WHERE type='table';
"""

# read sql query in pandas
tables = pd.read_sql(sql_query, conn)

# create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn) # Read table into dataframe
    globals()[f"df_{table_name}"] = df                    # Create dynamic dataframe name
    print(f"Created dataframe: df_{table_name}")

# Close connection
conn.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [18]:
# Print table names and column names
conn = sqlite3.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns['name'].tolist())

# Close connection
conn.close()


Table Name: db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


# 2.DATA CLEANING

In [ ]:
""" 
df_db_customer
"""

In [20]:
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,NaN
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,NaN
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,NaN
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,NaN
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,NaN


In [21]:
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
997,CUST-00977,Simran Gupta,Nepal,Haryana,Male,1975-11-14 19:47:27,NaN,672601.0
998,CUST-00978,Karan Rao,Nepal,Haryana,Female,1967-03-10 11:04:29,technology,604626.0
999,CUST-00979,Isha Khan,India,Gujarat,Male,1997-11-07 10:45:00,music,289223.0
1000,CUST-00005,Nisha Joshi,India,Tamil Nadu,Female,2002-06-23 17:09:32,fitness,672601.0
1001,CUST-00060,Kabir Khan,India,West Bengal,Male,1998-05-09 23:09:53,sports,122221.0


In [22]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   customerid  1002 non-null   str    
 1   name        1002 non-null   str    
 2   country     987 non-null    str    
 3   state       1002 non-null   str    
 4   gender      992 non-null    str    
 5   dob         1002 non-null   str    
 6   interests   894 non-null    str    
 7   pincode     981 non-null    float64
dtypes: float64(1), str(7)
memory usage: 62.8 KB


In [23]:
df_db_customer.isnull().sum()

customerid      0
name            0
country        15
state           0
gender         10
dob             0
interests     108
pincode        21
dtype: int64

In [24]:
df_db_customer["gender"].unique()

<StringArray>
['Male', 'Female', 'Women', 'Men', nan]
Length: 5, dtype: str

In [30]:
df_db_customer["country"].isnull().head(50)

0     False
1     False
2     False
3     False
4     False
5      True
6     False
7     False
8      True
9     False
10    False
11    False
12     True
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
40    False
41    False
42    False
43    False
44    False
45    False
46    False
47    False
48    False
49    False
Name: country, dtype: bool

In [35]:
df_db_customer[["name","country"]].iloc[0:40]

,name,country
0,keshav,India
1,raghav,India
2,lalita,India
3,mohan,India
4,mira,India
5,durga,NaN
6,mina,India
7,madan,India
8,maya,NaN
9,arjun,Nepal


In [ ]:
# so now what to do 
""" 
1.drop interest and pincode columns(becuz we don't need and many null value present)
2.rename name to customer_name (for better understanding)
3.standarized (in gender set only two  unique value i.e (male,female))
4.change data type of dob
5.set country as thier name 

"""

" \n1.drop interest and pincode columns(becuz we don't need and many null value present)\n2.rename name to customer_name (for better understanding)\n3.in gender set only two  unique value i.e (male,female)\n4.change data type of dob\n5.set country as thier name \n\n"

### A.drop interest and pindcode column

In [ ]:

df_db_customer.drop(columns=['interests','pincode'],inplace=True)

In [38]:
df_db_customer.head()

,customerid,name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00


 ### b.rename column ---name to customer_name

In [ ]:

df_db_customer.rename(columns={"name" : "customer_name"}, inplace= True)

In [42]:
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00


### c. standardize (gender column fixing)

In [47]:
df_db_customer['gender']=df_db_customer['gender'].replace({"Men":"Male", "Women":"Female"})

In [50]:
df_db_customer['gender'].unique()

<StringArray>
['Male', 'Female', nan]
Length: 3, dtype: str

### d.change datatype

In [43]:
df_db_customer['dob']=pd.to_datetime(df_db_customer['dob'])

In [44]:
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05


### e. Fix missing values

0       False
1       False
2       False
3       False
4       False
        ...  
997     False
998     False
999     False
1000    False
1001    False
Name: country, Length: 1002, dtype: bool

,customerid,customer_name,country,state,gender,dob
